### Data Ingestion

In [3]:
from langchain_core.documents import Document

In [4]:
doc=Document(
    page_content="this is the main text content I am using to create RAG",
    metadata={
        "source":"exmaple.txt",
        "pages":1,
        "author":"Pranav Gupta",
        "date_created":"2026-01-01"
        
    }
)
doc


Document(metadata={'source': 'exmaple.txt', 'pages': 1, 'author': 'Krish Naik', 'date_created': '2025-01-01'}, page_content='this is the main text content I am using to create RAG')

In [5]:
## Create a simple txt file
import os
os.makedirs("../data/text_files",exist_ok=True)

In [7]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}

for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text files created!")

✅ Sample text files created!


In [ ]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf", ## Pattern to match files  
    loader_cls= PyMuPDFLoader, ##loader class to use
    show_progress=False

)

pdf_documents=dir_loader.load()
pdf_documents

In [27]:
# Creating Data Chunks 

from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """
    Split documents into smaller chunks for better RAG performance.
    
    Parameters:
    - chunk_size: Maximum characters per chunk (adjust based on your LLM)
    - chunk_overlap: Characters to overlap between chunks (preserves context)
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, # Each chunk: ~1000 characters
        chunk_overlap=chunk_overlap, # 200 chars overlap for context
        length_function=len, # How to measure length
        separators=["\n\n", "\n", " ", ""] # Split hierarchy
    )
    # Actually split the documents
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show what a chunk looks like
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [8]:
### TextLoader
from langchain_community.document_loaders import TextLoader

loader=TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")
document=loader.load()
print(document)


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [9]:
### Directory Loader
from langchain_community.document_loaders import DirectoryLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt", ## Pattern to match files  
    loader_cls= TextLoader, ##loader class to use
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=False

)

documents=dir_loader.load()
documents

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.'),
 Document(metadata={'source': '../data/text_files/machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervise

In [23]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [20]:
class embeddingManager:
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        print(f"🔧 Initializing EmbeddingManager with model: {model_name}")
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        print(f"📦 Loading model: {self.model_name}...")
        self.model = SentenceTransformer(self.model_name)
        print(f"✅ Model loaded successfully!")
    
    def get_embeddings(self, texts: List[str]) -> np.ndarray:
        print(f"🔄 Generating embeddings for {len(texts)} text(s)...")
        embeddings = self.model.encode(texts, convert_to_numpy=True , show_progress_bar=True)
        print(f"✅ Embeddings generated! Shape: {embeddings.shape}")
        return embeddings
    
    def get_embedding_dimension(self) -> int:
        if self.model is None:
            raise ValueError("Model not loaded.")
        dimension = self.model.get_embedding_dimension()
        print(f"📊 Embedding dimension: {dimension}")
        return dimension
    
print("🚀 Instantiating EmbeddingManager...")
EmbeddingManager = embeddingManager()
print("✅ EmbeddingManager ready!")

🚀 Instantiating EmbeddingManager...
🔧 Initializing EmbeddingManager with model: all-MiniLM-L6-v2
📦 Loading model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5217.62it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded successfully!
✅ EmbeddingManager ready!


### Vector Store with ChromaDB

In [26]:
### Vector Store with ChromaDB

class ChromaDBVectorStore:
    def __init__(self, collection_name: str = "documents_collection", embedding_manager: embeddingManager = None):
        print(f"🔧 Initializing ChromaDBVectorStore with collection: {collection_name}")
        self.collection_name = collection_name
        self.embedding_manager = embedding_manager or embeddingManager()
        self.client = None
        self.collection = None
        self._initialize_chromadb()
    
    def _initialize_chromadb(self):
        print("📦 Initializing ChromaDB client...")
        self.client = chromadb.PersistentClient(path="./chroma_db")
        try:
            self.collection = self.client.get_collection(name=self.collection_name)
            print(f"⚠️ Collection '{self.collection_name}' already exists. Using existing collection.")
        except:
            print(f"✅ Creating new collection: {self.collection_name}")
            self.collection = self.client.create_collection(name=self.collection_name)
    
    def add_documents (self, documents: List[Any], embeddings: np.ndarray) :
        # Add documents and their embeddings to the vector store
        # Args:
        # documents: List of LangChain documents embeddings: Corresponding embeddings for the documents
        if len (documents) != len(embeddings) :
            raise ValueError ("Number of documents must match number of embeddings")
        print (f"Adding {len (documents)} documents to vector store...")
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            # Prepare metadata
            metadata = dict (doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append (metadata)
            # Document content
            documents_text.append(doc.page_content)
            # Embedding
            embeddings_list.append(embedding.tolist())
        # Add to collection
        try:
            self.collection.add(
            ids=ids, 
            embeddings=embeddings_list, 
            metadatas=metadatas,
            documents=documents_text
            )
            print (f"Successfully added {len (documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count ()}")
        except Exception as e:
            print (f"Error adding documents to vector store: {e}")
            raise
vector_store = ChromaDBVectorStore()
vector_store

🔧 Initializing ChromaDBVectorStore with collection: documents_collection
🔧 Initializing EmbeddingManager with model: all-MiniLM-L6-v2
📦 Loading model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6736.31it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded successfully!
📦 Initializing ChromaDB client...
⚠️ Collection 'documents_collection' already exists. Using existing collection.
